<div style="text-align:center; padding:20px 0"><img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/></div>

# AfriCare Support Analytics

## Ressources Power BI — Guide de création complet

> Document de référence pour la création du dashboard Power BI **AfriCare Support** (4 pages, theme LIGHT, navigation par onglets horizontaux). Conçu comme un script vidéo pas à pas.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 4h à 5h |
| **Approche** | Mockup PowerPoint + backgrounds PNG importés dans Power BI |

### Objectif business

Transformer les analyses SQL et ML en un dashboard décisionnel **4 pages** permettant à M. Kouamé de piloter : santé opérationnelle · performance SLA · classement agents · alertes ML préventives.

**Question finale à laquelle le dashboard doit répondre** :

> *"Que doit faire AfriCare dans les 90 prochains jours ?"*

---

## 1. Sources de données (4 fichiers)

### Fichiers à importer dans Power BI

| Fichier CSV | URL | Renommer en | Volume |
|---|---|---|---|
| Tickets nettoyés | `support_clean_analytics.csv` | `fact_tickets` | 15 287 lignes |
| Scores ML | `tickets_risque_scores.csv` | *(garder)* | 3 058 lignes |
| Agents | `agents.csv` | `dim_agents` | 12 lignes |
| Catégories | `categories.csv` | `dim_categories` | 10 lignes |

### URLs GitHub raw

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/corrige/outputs/support_clean_analytics.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/corrige/outputs/tickets_risque_scores.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/data/agents.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/customer_support_analytics/data/categories.csv
```

### Schéma simplifié

```
                    dim_agents (12)
                         |
dim_categories ----- fact_tickets ----- Calendrier
   (10)               (15 287)          (1 ligne/jour)
                         |
              tickets_risque_scores
                     (3 058)
```

### Import

1. Power BI Desktop → **Obtenir des données → Web**
2. Coller chaque URL dans la fenêtre web
3. Cliquer **Transformer les données** (pour vérifier les types)
4. Renommer les tables dans le panneau Requêtes (clic droit → Renommer)
5. Vérifier les types (notamment `sla_breach` et `in_backlog` en **Nombre entier** — 0/1)
6. **Accueil → Fermer & appliquer**

---

## 2. Désactiver Auto Date/Time (obligatoire)

Avant toute manipulation du modèle :

**Fichier → Options → Chargement des données (Fichier actuel) → DÉCOCHER "Date/heure automatique pour le fichier actuel"**

Sans cette étape, Power BI crée des tables `LocalDateTable_*` parasites qui polluent le modèle et empêchent `PREVIOUSMONTH` / `SAMEPERIODLASTYEAR` de fonctionner correctement.

---

## 3. Modèle de données — Schéma en étoile

### Architecture

```
                    dim_agents (12 lignes)
                         |
dim_categories --- fact_tickets --- Calendrier
   (10 lignes)       (15 287 lignes)         (1 ligne/jour)
                         |
                 tickets_risque_scores
                       (3 058 lignes)
```

### Relations à configurer (4 relations)

| Table (N) | Colonne | Table (1) | Colonne | Cardinalité | Direction |
|---|---|---|---|---|---|
| `fact_tickets` | `agent_id` | `dim_agents` | `agent_id` | N→1 | Single |
| `fact_tickets` | `category_id` | `dim_categories` | `category_id` | N→1 | Single |
| `fact_tickets` | `created_at` (date) | `Calendrier` | `Date` | N→1 | Single |
| `tickets_risque_scores` | `ticket_id` | `fact_tickets` | `ticket_id` | 1→1 | Single |

### ⚠️ Si `created_at` contient une heure

Dans Power Query, créer une colonne calculée `date_created` :

```m
= Date.From([created_at])
```

Puis lier `date_created → Calendrier[Date]` au lieu de `created_at`.

### Vérifications

- Aucune relation M:M bidirectionnelle
- Aucun chemin ambigu
- Pas de doublon de relation entre 2 tables

---

## 4. Table Calendrier

Modélisation → **Nouvelle table** → coller :

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "MMM", "fr-FR"),
    "Mois_Nom_Long", FORMAT([Date], "MMMM", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "YYYY-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR"),
    "Est_Weekend",   IF(WEEKDAY([Date],2) >= 6, 1, 0)
)
```

**Marquer comme table de dates** : clic droit `Calendrier` → *Marquer comme table de dates* → colonne `Date`.

### Tri du mois nominal par numéro

Pour que Janvier apparaisse avant Février sur les axes des graphiques :

1. Sélectionner la colonne `Mois_Nom` dans le panneau Données
2. Onglet **Outils de colonne** → **Trier par colonne** → choisir `Mois_Num`

Idem pour `Mois_Nom_Long`.

---

## 5. Table _Mesures (placeholder)

Modélisation → **Nouvelle table** → coller :

```dax
_Mesures = {BLANK()}
```

Puis dans le panneau Données :
- Clic droit sur la colonne `Value` → **Masquer**
- La table `_Mesures` apparaît avec une icône calculatrice (juste pour les mesures)

> **Astuce** : le `_` au début du nom force Power BI à afficher cette table en premier dans la liste alphabétique.

---

## 6. Design system — AfriCare Support

### ⚠️ Spécificité vs autres rapports DPL

Ce rapport utilise un **design clair** (fond blanc/gris pâle) — à la différence des dashboards e-commerce (navy) ou hôtelier (or/noir). Chaque onglet porte une couleur dédiée qui signale visuellement la section en cours.

### Palette AfriCare

| Usage | Couleur | Hex |
|---|---|---|
| Fond principal | Blanc | `#FFFFFF` |
| Fond section | Gris très clair | `#F5F5F7` |
| Bordure carte | Gris pâle | `#E5E7EB` |
| **Onglet Overview** (actif) | Violet | `#5B21B6` |
| **Onglet Performance SLA** (actif) | Orange | `#F97316` |
| **Onglet Agents** (actif) | Vert | `#10B981` |
| **Onglet Alertes ML** (actif) | Rouge | `#EF4444` |
| Onglet inactif | Gris texte | `#6B7280` |
| Performance positive / Résolu | Vert | `#10B981` |
| Alerte / Escaladé | Rouge | `#EF4444` |
| Vigilance / Backlog | Orange | `#F59E0B` |
| En cours | Bleu | `#3B82F6` |
| Fermé / Neutre | Gris moyen | `#9CA3AF` |
| Jaune (alerte ML niveau 2) | Jaune | `#FBBF24` |
| Texte principal | Quasi-noir | `#111827` |
| Texte secondaire | Gris moyen | `#6B7280` |

### Typographie

| Usage | Police | Taille |
|---|---|---|
| **Titres pages (serif)** | **Georgia** regular | 22-26px |
| Navigation onglets | Segoe UI medium | 14px |
| Valeurs KPI | Segoe UI bold | 32-38px |
| Labels KPI | Segoe UI regular | 11-12px |
| Texte tableaux | Segoe UI regular | 10-11px |

### Principes

- Fond clair global → contraste élevé pour lecture rapide en environnement bureau
- Chaque carte KPI : bordure supérieure colorée 3px selon la nature du KPI
- Les bars par catégorie de breach sont colorées **par seuil** (rouge ≥ 50%, orange 30-50%, vert < 30%)
- Le choix de Georgia (serif) pour les titres donne un look "publication / rapport métier" élégant

---

## 7. Mockup PowerPoint et backgrounds

### Livrables fournis

| Fichier | Rôle |
|---|---|
| `mockup_africare_powerbi.pptx` | Maquette statique 4 slides — référence design |
| `bg-01-overview.png` | Background Page 1 (onglet Overview violet actif) |
| `bg-02-performance-sla.png` | Background Page 2 (onglet SLA orange actif) |
| `bg-03-agents.png` | Background Page 3 (onglet Agents vert actif) |
| `bg-04-alertes-ml.png` | Background Page 4 (onglet Alertes ML rouge actif) |

Les 4 PNG vierges (2001×1125 px) contiennent déjà :
- Header complet (logo + 4 onglets + 2 slicers date)
- Onglet correspondant surligné avec sa couleur dédiée
- Cartes vides avec leur accent border coloré
- Titre de page en Georgia avec petit underline coloré

### Procédure d'import du background

Pour chaque page Power BI :

1. Sélectionner la page (onglet en bas)
2. Volet **Format de la page** (zone vide cliquée)
3. **Arrière-plan de page** → activer
4. **Image** → **Parcourir** → choisir le PNG correspondant
5. **Ajustement de l'image** : choisir **Ajuster**
6. **Transparence** : 0% (par défaut)

### Bénéfice de cette approche

- Tu évites de construire le bandeau header à la main (logo, onglets, slicers stylisés)
- Le rendu visuel est immédiatement professionnel
- Tu poses les visuels Power BI dans les zones vides — alignement et accent borders sont déjà en place

### Slicers et boutons sur le PNG

Le PNG contient l'**aspect visuel** des slicers et des boutons d'onglet, mais ce sont juste des images.

Pour rendre les onglets cliquables :
- Insertion → **Bouton transparent** posé exactement sur la zone de chaque onglet du PNG
- Format → Action → **Type : Navigation de page** → cible = page correspondante

Pour les slicers date fonctionnels :
- Insertion → **Segment** posé exactement sur les 2 zones du PNG
- Champ : `Calendrier[Date]`
- Mode : **Avant** pour le premier, **Après** pour le second
- Format → **En-tête** désactivé (pour ne pas dupliquer le label déjà présent dans le PNG)
- Synchroniser sur les 4 pages

---

## 8. Architecture de navigation — Onglets horizontaux

### ⚠️ Différence avec les autres rapports DPL

Ici la navigation est **horizontale en haut de page** (pas de sidebar latérale). Chaque onglet est un bouton transparent avec sa couleur dédiée quand il est actif (déjà visible dans le PNG de fond).

### Bandeau supérieur (présent sur les 4 pages via PNG background)

**Zone gauche** : Logo avatar AfriCare (cercle violet 0,5" avec initiales "AC") + texte `AfriCare Support` en Segoe UI bold 16px.

**Zone centrale** : 4 onglets de navigation, alignés horizontalement.

| Ordre | Label | Couleur active | Page cible |
|---|---|---|---|
| 1 | Overview | Violet `#5B21B6` | Page 1 |
| 2 | Performance SLA | Orange `#F97316` | Page 2 |
| 3 | Agents | Vert `#10B981` | Page 3 |
| 4 | Alertes ML | Rouge `#EF4444` | Page 4 |

**Zone droite** : 2 slicers date range — `01/01/2022` et `31/12/2024`.

### Implémentation des boutons de navigation

Pour chaque onglet (4 boutons par page, 16 boutons au total) :

1. **Insertion → Boutons → Vide**
2. Le positionner exactement sur la zone de l'onglet du PNG
3. Format → **Action** activée → **Type : Navigation de page** → cible = page correspondante
4. Format → **Texte du bouton** : désactivé (le texte est déjà dans le PNG)
5. Format → **Forme** : remplissage transparent, bordure transparente

**Astuce** : sélectionner les 4 boutons sur la Page 1 (Ctrl+clic) → Ctrl+C → coller sur les 3 autres pages.

---

## 9. Slicers globaux — Date range

### Un seul groupe : Date range

Dans le bandeau supérieur droit des 4 pages :

| Slicer | Champ | Style | Valeurs par défaut |
|---|---|---|---|
| **Date début** | `Calendrier[Date]` | **Avant** (mode date) | 01/01/2022 |
| **Date fin** | `Calendrier[Date]` | **Après** | 31/12/2024 |

### Style

- Fond transparent (le PNG fournit déjà la zone visuelle)
- Bordure transparente
- Texte en Segoe UI regular 13px
- En-tête désactivé

### Synchronisation

Affichage → **Synchroniser les segments** → cocher les 4 pages dans les colonnes **Visible** et **Filtre**.

> 💡 **Pas d'autre slicer global** dans ce rapport. Les filtres spécifiques par page (canal, catégorie, agent) peuvent être ajoutés au volet Filtres mais ne sont pas exposés en slicer visible.

---

## 10. Page 1 — Overview

**Background** : `bg-01-overview.png` (onglet violet actif)

**Titre** : `Vue d'ensemble — Support AfriCare` 

### Ligne 1 : 6 KPI cards

Chaque carte se pose dans la zone correspondante du PNG. Bordure supérieure colorée 3px déjà présente dans le PNG.

| # | Label | Mesure | Valeur attendue | Pastille | Couleur top |
|---|---|---|---|---|---|
| 1 | Tickets total | `[Nb Tickets Total]` | **15 287** | ▲ +24,9% vs N-1 | 🔘 Gris `#9CA3AF` |
| 2 | Taux SLA breach | `[Taux SLA Breach %]` | **47,3%** | ▲ 0,0pp vs N-1 | 🔴 Rouge `#EF4444` |
| 3 | Backlog | `[Taux Backlog %]` | **7,4%** | `[Nb Backlog]` = 1 126 tickets | 🟠 Orange `#F59E0B` |
| 4 | Délai 1ère réponse | `[Delai Premiere Reponse Moy (h)]` | **36,8h** | ▼ +0,1h vs N-1 | 🔴 Rouge `#EF4444` |
| 5 | CSAT moyen | `[CSAT Moyen]` | **4,12** | ★★★★☆ /5 | 🟢 Vert `#10B981` |
| 6 | Réouverture | `[Taux Reouverture %]` | **7,7%** | `[Nb Tickets Rouverts]` = 1 181 | 🔘 Gris `#9CA3AF` |

### Ligne 2 : 2 visuels côte à côte

**Gauche — Évolution taux SLA breach (%)** (courbe lissée)
- Visuel : **Graphique en courbes**
- Axe X : `Calendrier[Mois_Nom]` (janv à déc)
- Axe Y : `[Taux SLA Breach %]`
- Couleur : rouge `#EF4444`, ligne épaisse 3px, courbe lissée
- Ligne constante horizontale rose pointillée à 30% (cible SLA)
- Format : 0,0%

**Droite — Répartition des statuts** (donut)
- Visuel : **Graphique en anneau**
- Légende : `fact_tickets[statut]`
- Valeurs : `[Nb Tickets Total]`
- Data labels : % intérieur (ligne de rappel pour Résolu)
- Palette :
  - Résolu vert `#10B981`
  - Escaladé rouge `#EF4444`
  - En cours bleu `#3B82F6`
  - Backlog orange `#F59E0B`
  - Fermé gris `#9CA3AF`
- Résultat : Résolu 52,78% · Escaladé 21,74% · En cours 15% · Backlog 7,37% · Fermé ~3%

### Ligne 3 : 2 visuels côte à côte

**Gauche — Taux SLA breach par catégorie (%)** (bar chart horizontal)
- Visuel : **Histogramme à barres** trié DESC
- Axe Y : `dim_categories[nom_categorie]`
- Axe X : `[Taux SLA Breach %]`
- **Couleur par formule** : `[Couleur Breach Agent]` (rouge ≥ 50%, orange 30-50%, vert < 30%)
- Ligne constante verticale rouge pointillée à 50% (seuil contractuel)
- Top observé : Demande info 68,6% · Facturation 59,4% · Remboursement 52,6% · Livraison retardée 51,6% · Qualité produit 51,1% · Panne technique 39,4% · Problème paiement 37,8% · Fraude signalée 25,5% · Escalade juridique 23,2% · Accès compte 20,7%

**Droite — Tickets par pays** (bar chart horizontal)
- Visuel : **Histogramme à barres** trié DESC
- Axe Y : `fact_tickets[pays]` (Top 5)
- Axe X : `[Nb Tickets Total]`
- Couleur unique : violet `#5B21B6`
- Top 5 : CI ~4 000 · Sénégal ~3 000 · France ~2 300 · Maroc ~2 100 · Ghana ~2 000

---

## 11. Page 2 — Performance SLA

**Background** : `bg-02-performance-sla.png` (onglet orange actif)

**Titre** : `Performance SLA — par catégorie & canal` 

### Ligne 1 : 4 KPI cards + 1 jauge

| # | Label | Mesure | Valeur | Complément | Couleur top |
|---|---|---|---|---|---|
| 1 | Breach global | `[Taux SLA Breach %]` | **47,3%** | Objectif : 30% | 🔴 Rouge |
| 2 | Breach SLA strict | `[Taux Breach SLA Strict %]` | **42,9%** | Risque contractuel (texte rouge) | 🔴 Rouge |
| 3 | Dépassement moyen | `[Depassement SLA Moy (h)]` | **+121h** | Sur tickets en breach | 🟠 Orange |
| 4 | Ratio SLA moyen | `[Ratio SLA Moyen]` | **1,00×** | Résolution / SLA contractuel | 🟠 Orange |

**À droite des 4 cartes — Jauge circulaire % breach vs objectif**
- Visuel : **Jauge** (natif) ou **KPI circulaire** custom
- Valeur : `[Taux SLA Breach %]` → 47,3%
- Min : 0% · Max : 100% · Cible : 30%
- Zones colorées : 0-30% vert, 30-50% orange, 50-100% rouge
- Titre : "% breach vs objectif 30%"

### Ligne 2 : 2 visuels côte à côte

**Gauche — Taux SLA breach par catégorie (%)** (réplication Page 1)
- **Identique à celui de la Page 1** (même mesure `Couleur Breach Agent`, même tri DESC, même ligne cible verticale à 50%)
- Réplication volontaire : permet la lecture autonome de cette page

**Droite — Répartition des statuts** (donut, réplication)
- **Identique à celui de la Page 1** (même palette, même mesure)
- Version légèrement plus grande (met en évidence le 3,12% Fermé visible)

### Ligne 3 : 2 visuels côte à côte

**Gauche — Heatmap Catégorie × Mois** (matrice)
- Visuel : **Matrice** native
- Lignes : `dim_categories[nom_categorie]`
- Colonnes : `Calendrier[Mois_Num]` (janv à déc)
- Valeurs : `[Taux SLA Breach %]` (format 0,0%)
- **Formatage conditionnel sur la couleur de fond** : règle *Par dégradé*
  - Min 20% vert `#86EFAC`
  - Milieu 45% jaune `#FEF08A`
  - Max 70% rouge `#FCA5A5`
- Supprimer totaux lignes/colonnes pour garder uniquement les cellules
- Lecture : on voit les catégories problématiques (Demande info, Facturation) en rouge quasi tous les mois

**Droite — Taux SLA breach par canal (%)** (bar chart vertical)
- Visuel : **Histogramme à colonnes**
- Axe X : `fact_tickets[canal]`
- Axe Y : `[Taux SLA Breach %]`
- Couleur unique : violet foncé `#5B21B6`
- Résultat : Téléphone 48,0% · Chat 47,9% · App mobile 47,4% · Web form 46,6% · Email 46,2%
- **Insight** : les 5 canaux sont quasi homogènes → le canal n'est PAS un facteur différenciant

---

## 12. Page 3 — Agents

**Background** : `bg-03-agents.png` (onglet vert actif)

**Titre** : `Performance agents — classement & matrice` 

### Layout en 2 blocs

#### Gauche : Tableau Classement agents

**Visuel : Table** (native) avec 7 colonnes et 12 lignes.

| Colonne | Champ / Mesure | Formatage |
|---|---|---|
| Agent | `dim_agents[nom_agent]` | Segoe UI regular |
| Tier | `dim_agents[tier]` | **Badge coloré** : Tier 1 gris clair, Tier 2 vert clair, Tier 3 violet clair |
| Rg SLA | `[Rang Agent SLA]` | Bleu centré |
| Breach | `[Taux Breach Agent %]` | **Texte coloré** via `[Couleur Breach Agent]` |
| CSAT | `[CSAT Moyen]` | Format 0,0 |
| Écart | `[Ecart CSAT Agent]` | Format 0,00 (différence vs moyenne globale) |
| Segment | `[Segment Agent]` | Badge jaune clair "Coach" |

### Les 12 agents observés (triés alphabétiquement)

| Agent | Tier | Breach | CSAT | Écart |
|---|---|---|---|---|
| Aissatou Ba | Tier 1 | 48,3% | 3,9 | -0,07 |
| Aminata Diallo | Tier 1 | 43,4% | 4,2 | 0,00 |
| Aya Touré | Tier 2 | 46,3% | 4,7 | -0,10 |
| Fatou Sow | Tier 2 | 39,9% | 4,5 | -0,10 |
| Ibrahim Coulibaly | Tier 2 | 42,2% | 4,3 | -0,07 |
| Jean-Marc Dubois | Tier 3 | 40,0% | 4,8 | -0,12 |
| Kofi Mensah | Tier 1 | 53,6% | 3,8 | -0,10 |
| **Moussa Kone** | Tier 1 | **60,1%** | 3,5 | -0,14 |
| Nadia Benhaddou | Tier 1 | 48,3% | 4,0 | -0,14 |
| Olivier Martin | Tier 3 | 44,8% | 4,6 | -0,08 |
| **Ramatou Diaby** | Tier 1 | **55,1%** | 3,6 | -0,08 |
| Samuel Acheampong | Tier 2 | 45,1% | 4,4 | -0,02 |

**3 agents Tier 1 en alerte (> 50% breach)** : Moussa Kone · Ramatou Diaby · Kofi Mensah.

#### Droite : Matrice performance — Breach vs CSAT

**Visuel : Nuage de points (Scatter)**
- Axe X : `[Taux Breach Agent %]` (de 40% à 60%)
- Axe Y : `[CSAT Moyen]` (de 3,4 à 4,8)
- Détails : `dim_agents[nom_agent]`
- **Légende (couleur)** : `dim_agents[tier]`
  - Tier 1 → gris pâle `#9CA3AF`
  - Tier 2 → vert pâle `#86EFAC`
  - Tier 3 → violet pâle `#C4B5FD`
- Taille bulle : valeur fixe ou `[Nb Tickets Agent]`
- Opacité 60% pour voir les superpositions

> 💡 **Pas de lignes de quadrant dans ce rapport**. La lecture se fait visuellement : les bulles en haut-gauche (faible breach + CSAT élevé) = top performers Tier 3 ; les bulles en bas-droite (fort breach + CSAT bas) = coaching prioritaire.

---

## 13. Page 4 — Alertes ML

**Background** : `bg-04-alertes-ml.png` (onglet rouge actif)

**Titre** : `Alertes ML — Tickets à risque détectés` 

### Ligne 1 : 5 KPI cards

Chaque carte a un fond légèrement teinté de sa couleur de criticité .

| # | Label | Mesure | Valeur | Complément | Couleur |
|---|---|---|---|---|---|
| 1 | Alertes ROUGE | `[Nb Alertes ROUGE]` | **314** | Intervention immédiate | 🔴 Rouge `#EF4444` (fond rosé) |
| 2 | Alertes ORANGE | `[Nb Alertes ORANGE]` | **1 542** | À surveiller | 🟠 Orange `#F59E0B` (fond orangé pâle) |
| 3 | Alertes JAUNE | `[Nb Alertes JAUNE]` | **931** | Attention | 🟡 Jaune `#FBBF24` (fond jaune pâle) |
| 4 | Score risque moyen | `[Score Risque Max]` | **0,50** | Sur tickets alertés | 🟣 Violet `#8B5CF6` |
| 5 | Recall modèle | `[Recall Modele]` | **79,6%** | Objectif > 75% ✓ (checkmark vert) | 🟢 Vert `#10B981` |

### Ligne 2 : Tableau tickets + Distribution alertes

**Gauche (largeur 60%) — Liste des tickets alertés** (triés par score DESC)
- Visuel : **Table** native
- Pré-filtrée sur `niveau_alerte` contient "ROUGE" (filtre de page)
- Trié par `score_risque` DESC

| Colonne | Champ | Formatage |
|---|---|---|
| Ticket | `tickets_risque_scores[ticket_id]` | Regular |
| Pays | `fact_tickets[pays]` | Regular |
| Canal | `fact_tickets[canal]` | Regular |
| Agent | `dim_agents[nom_agent]` | Regular |
| Score | `tickets_risque_scores[score_risque]` | Format 0,00 |
| Priorite | `fact_tickets[priorite]` | Entier |
| Niveau Alerte | `tickets_risque_scores[niveau_alerte]` | **Badge rouge** texte blanc |

**Top 5 tickets observés** :
- TKT014857 · Cameroun · Email · Jean-Marc Dubois · 0,84 · 5 · ROUGE
- TKT012691 · CI · Téléphone · Moussa Kone · 0,82 · 4 · ROUGE
- TKT013209 · Ghana · Téléphone · Jean-Marc Dubois · 0,81 · 5 · ROUGE
- TKT013797 · Maroc · Chat · Moussa Kone · 0,81 · 4 · ROUGE
- TKT014745 · France · Web form · Moussa Kone · 0,81 · 5 · ROUGE

**Droite (largeur 40%) — Distribution des niveaux d'alerte** (bar vertical)
- Visuel : **Histogramme à colonnes**
- Axe X : `tickets_risque_scores[niveau_alerte]`
- Axe Y : `[Nb Alertes Total]`
- **Couleur par catégorie** :
  - ORANGE → orange pâle `#FDBA74`
  - JAUNE → jaune doré `#EAB308`
  - ROUGE → rouge pâle `#FCA5A5`
  - VERT → vert `#10B981`
- Ordre observé : ORANGE 1542 · JAUNE 931 · ROUGE 314 · VERT 271

### Ligne 3 : Top agents + Jauge score

**Gauche — Top agents — alertes ROUGE** (bar horizontal)
- Visuel : **Histogramme à barres** trié DESC
- Axe Y : `dim_agents[nom_agent]` (Top 5)
- Axe X : `[Nb Alertes ROUGE]`
- Couleur : rouge `#EF4444` pour les 3 premiers, orange `#F59E0B` pour les suivants
- **Top 5** : Moussa Kone 66 · Ramatou Diaby 52 · Aissatou Ba 36 · Kofi Mensah 34 · Jean-Marc Dubois 26

**Droite — Score risque moyen** (jauge horizontale custom)
- Construction : 4 rectangles colorés + marqueur vertical noir
- Échelle 0 à 1 avec 4 zones colorées :
  - 0 à 0,25 : vert `#10B981`
  - 0,25 à 0,45 : jaune `#FBBF24`
  - 0,45 à 0,70 : orange `#F59E0B`
  - 0,70 à 1 : rouge `#EF4444`
- Marqueur vertical noir positionné à `[Score Risque Max]` = 0,50
- Label sous la jauge : "**Score moyen : 0,50**" en Segoe UI bold

---

## 14. Mesures DAX — Dossier `KPIs_GLOBAUX` (15 mesures)

Toutes les mesures sont créées dans la table `_Mesures` avec le **dossier d'affichage** = `KPIs_GLOBAUX`.

```dax
Nb Tickets Total = COUNTROWS(fact_tickets)

Nb Tickets Resolus =
CALCULATE(
    COUNTROWS(fact_tickets),
    fact_tickets[statut] = "Resolu"
)

Taux Resolution % = DIVIDE([Nb Tickets Resolus], [Nb Tickets Total], 0)

Nb SLA Breach =
CALCULATE(
    COUNTROWS(fact_tickets),
    fact_tickets[sla_breach] = 1
)

Taux SLA Breach % = DIVIDE([Nb SLA Breach], [Nb Tickets Total], 0)

l

Taux Backlog % = DIVIDE([Nb Backlog], [Nb Tickets Total], 0)

Nb Escaladé =
CALCULATE(
    COUNTROWS(fact_tickets),
    fact_tickets[statut] = "Escalade"
)

Taux Escaladé % = DIVIDE([Nb Escaladé], [Nb Tickets Total], 0)

Delai Premiere Reponse Moy (h) = AVERAGE(fact_tickets[first_response_heures])

Delai Resolution Moy (h) = AVERAGE(fact_tickets[resolution_heures])

CSAT Moyen =
AVERAGEX(
    FILTER(fact_tickets, NOT(ISBLANK(fact_tickets[csat]))),
    fact_tickets[csat]
)

Nb Tickets Rouverts =
CALCULATE(
    COUNTROWS(fact_tickets),
    fact_tickets[reouvert] = 1
)

Taux Reouverture % = DIVIDE([Nb Tickets Rouverts], [Nb Tickets Total], 0)

Nb Tickets a Risque = COUNTROWS(tickets_risque_scores)

Taux Risque % = DIVIDE([Nb Tickets a Risque], [Nb Tickets Total], 0)
```

### Format strings recommandés

- Mesures `Nb...` → `#,0`
- Mesures `Taux ... %` → `0.0%`
- `CSAT Moyen` → `0.00`
- `Delai ... (h)` → `0.0`

---

## 15. Mesures DAX — Dossier `SLA` (6 mesures avancées)

```dax
Nb Breach SLA Strict =
CALCULATE(
    COUNTROWS(fact_tickets),
    fact_tickets[sla_breach] = 1,
    dim_categories[sla_strict] = 1
)

Taux Breach SLA Strict % =
DIVIDE(
    [Nb Breach SLA Strict],
    CALCULATE(COUNTROWS(fact_tickets), dim_categories[sla_strict] = 1),
    0
)

Depassement SLA Moy (h) =
CALCULATE(
    AVERAGE(fact_tickets[depassement_heures]),
    fact_tickets[sla_breach] = 1
)

Ratio SLA Moyen =
AVERAGEX(
    fact_tickets,
    DIVIDE(fact_tickets[resolution_heures], fact_tickets[sla_heures])
)

Taux SLA Breach % PM =
CALCULATE(
    [Taux SLA Breach %],
    PREVIOUSMONTH(Calendrier[Date])
)

Evolution Breach pp =
([Taux SLA Breach %] - [Taux SLA Breach % PM]) * 100
```

### Format strings

- `Nb Breach SLA Strict` → `#,0`
- `Taux Breach SLA Strict %`, `Taux SLA Breach % PM` → `0.0%`
- `Depassement SLA Moy (h)` → `0`
- `Ratio SLA Moyen` → `0.00"x"`
- `Evolution Breach pp` → `+0.0;-0.0;0`

---

## 16. Mesures DAX — Dossier `Agents` (7 mesures)

```dax
Taux Breach Agent % = [Taux SLA Breach %]

Taux Escaladé Agent % = [Taux Escaladé %]

Ecart CSAT Agent =
VAR _agentCsat = [CSAT Moyen]
VAR _globalCsat = CALCULATE([CSAT Moyen], ALL(dim_agents))
RETURN _agentCsat - _globalCsat

Rang Agent SLA =
RANKX(
    ALL(dim_agents[agent_id]),
    CALCULATE([Taux SLA Breach %]),
    ,
    ASC
)

Rang Agent CSAT =
RANKX(
    ALL(dim_agents[agent_id]),
    CALCULATE([CSAT Moyen]),
    ,
    DESC
)

Couleur Breach Agent =
VAR _tx = [Taux SLA Breach %]
RETURN
    SWITCH(
        TRUE(),
        _tx >= 0.50, "#EF4444",
        _tx >= 0.40, "#F59E0B",
        "#10B981"
    )

Segment Agent =
VAR _ecart = [Ecart CSAT Agent]
VAR _breach = [Taux SLA Breach %]
RETURN
    SWITCH(
        TRUE(),
        _breach >= 0.55, "Urgent",
        _breach >= 0.40 && _ecart < 0, "Coach",
        _ecart >= 0 && _breach < 0.40, "Top",
        "Coach"
    )

Couleur Tier Agent =
SWITCH(
    SELECTEDVALUE(dim_agents[tier]),
    "Tier 1", "#F3F4F6",   -- gris très pâle (badge fond)
    "Tier 2", "#DCFCE7",   -- vert pâle
    "Tier 3", "#EDE9FE",   -- violet pâle
    "#F3F4F6"               -- défaut : gris pâle
)

Couleur Texte Tier Agent =
SWITCH(
    SELECTEDVALUE(dim_agents[tier]),
    "Tier 1", "#6B7280",   -- gris (texte sur fond gris pâle)
    "Tier 2", "#10B981",   -- vert
    "Tier 3", "#5B21B6",   -- violet
    "#6B7280"
)

Couleur Segment Agent =
SWITCH(
    [Segment Agent],
    "Urgent", "#FEE2E2",   -- rouge pâle (alerte)
    "Coach",  "#FEF3C7",   -- jaune pâle (coaching)
    "Top",    "#DCFCE7",   -- vert pâle (performance)
    "#FEF3C7"
)

Couleur Texte Segment Agent =
SWITCH(
    [Segment Agent],
    "Urgent", "#EF4444",   -- rouge intense
    "Coach",  "#F59E0B",   -- orange
    "Top",    "#10B981",   -- vert
    "#F59E0B"
)
```

> **Note importante sur RANKX** : `ALL(dim_agents[agent_id])` est obligatoire pour ranker sur tous les agents, sinon le rang reste à 1 (filtre implicite du contexte de ligne).

### Utilisation de `Couleur Breach Agent`

Dans le visuel bar chart de la Page 1 et le tableau de la Page 3 :

1. Volet Format → **Couleurs des données** (ou **Couleur de la police** pour le tableau)
2. Cliquer sur **fx** (formatage conditionnel)
3. **Mettre en forme par : Valeur du champ**
4. Champ basé sur : `Couleur Breach Agent`

---

## 17. Mesures DAX — Dossier `Alertes ML` (6 mesures)

Le filtre utilise `SEARCH("texte", colonne, 1, 0) > 0` pour matcher "ROUGE -- Intervention immediate" quel que soit le suffixe.

```dax
Nb Alertes Total = COUNTROWS(tickets_risque_scores)

Nb Alertes ROUGE =
CALCULATE(
    COUNTROWS(tickets_risque_scores),
    SEARCH("ROUGE", tickets_risque_scores[niveau_alerte], 1, 0) > 0
)

Nb Alertes ORANGE =
CALCULATE(
    COUNTROWS(tickets_risque_scores),
    SEARCH("ORANGE", tickets_risque_scores[niveau_alerte], 1, 0) > 0
)

Nb Alertes JAUNE =
CALCULATE(
    COUNTROWS(tickets_risque_scores),
    SEARCH("JAUNE", tickets_risque_scores[niveau_alerte], 1, 0) > 0
)

Score Risque Max =
CALCULATE(
    AVERAGE(tickets_risque_scores[score_risque]),
    SEARCH("VERT", tickets_risque_scores[niveau_alerte], 1, 0) = 0
)

Titre Alertes =
"Alertes ML — " & [Nb Alertes ROUGE] & " ROUGE · " &
[Nb Alertes ORANGE] & " ORANGE · " & [Nb Alertes JAUNE] & " JAUNE"
```

### Mesure Recall (issue du Notebook 5)

```dax
Recall Modele = 0.796
```

Format : `0.0%` — la valeur est figée à 79,6% (issue de l'évaluation du modèle ML dans le Notebook 5).

### Format strings

- Mesures `Nb...` → `#,0`
- `Score Risque Max` → `0.00`
- `Recall Modele` → `0.0%`

In [ ]:
---

## 18. Mesures DAX — Dossier `Variations vs N-1` (6 mesures)

```dax
Tickets vs N-1 =
VAR _curr = [Nb Tickets Total]
VAR _prev = CALCULATE([Nb Tickets Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_curr - _prev, _prev)
VAR _fmt = FORMAT(_pct, "+0.0%;-0.0%")
RETURN
    SWITCH(
        TRUE(),
        _pct > 0, UNICHAR(9650) & " " & _fmt & " vs N-1",
        _pct < 0, UNICHAR(9660) & " " & _fmt & " vs N-1",
        UNICHAR(9650) & " 0,0% vs N-1"
    )

SLA Breach vs N-1 =
VAR _curr = [Taux SLA Breach %]
VAR _prev = CALCULATE([Taux SLA Breach %], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = (_curr - _prev) * 100
VAR _fmt = FORMAT(_delta, "+0.0;-0.0") & "pp"
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " " & _fmt & " vs N-1",
        _delta < 0, UNICHAR(9660) & " " & _fmt & " vs N-1",
        UNICHAR(9650) & " 0,0pp vs N-1"
    )

Delai vs N-1 =
VAR _curr = [Delai Premiere Reponse Moy (h)]
VAR _prev = CALCULATE([Delai Premiere Reponse Moy (h)], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _curr - _prev
VAR _fmt = FORMAT(_delta, "+0.0;-0.0") & "h"
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " " & _fmt & " vs N-1",
        _delta < 0, UNICHAR(9660) & " " & _fmt & " vs N-1",
        UNICHAR(9650) & " 0,0h vs N-1"
    )

Nb Categories >50% Breach =
COUNTROWS(
    FILTER(
        VALUES(dim_categories[nom]),
        [Taux SLA Breach %] > 0.50
    )
)

Alerte Contractuelle =
VAR _nb = [Nb Categories >50% Breach]
RETURN
    IF(
        _nb > 0,
        "Billing SLA strict : " & _nb & " categories depassent 50% de breach. Risque de penalites contractuelles.",
        BLANK()
    )
```

In [ ]:
## 19. Mesures DAX — Dossier `Couleurs Variation` (3 mesures)

```dax 

Couleur Tickets vs N-1 =
VAR _pct = DIVIDE(
    [Nb Tickets Total] - CALCULATE([Nb Tickets Total], SAMEPERIODLASTYEAR(Calendrier[Date])),
    CALCULATE([Nb Tickets Total], SAMEPERIODLASTYEAR(Calendrier[Date]))
)
RETURN
    SWITCH(TRUE(),
        _pct > 0, "#10B981",   -- Vert : croissance
        _pct < 0, "#EF4444",   -- Rouge : décroissance
        "#9CA3AF"               -- Gris : stable
    )

Couleur SLA Breach vs N-1 =
VAR _delta = [Taux SLA Breach %] - CALCULATE([Taux SLA Breach %], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN
    SWITCH(TRUE(),
        _delta > 0, "#EF4444",   -- Rouge : plus de breach = MAUVAIS (inversé)
        _delta < 0, "#10B981",   -- Vert : moins de breach = bon
        "#9CA3AF"
    )

Couleur Delai vs N-1 =
VAR _delta = [Delai Premiere Reponse Moy (h)] - CALCULATE([Delai Premiere Reponse Moy (h)], SAMEPERIODLASTYEAR(Calendrier[Date]))
RETURN
    SWITCH(TRUE(),
        _delta > 0, "#EF4444",   -- Rouge : délai plus long = MAUVAIS (inversé)
        _delta < 0, "#10B981",   -- Vert : délai plus court = bon
        "#9CA3AF"
    )
    
```

---

## 19. Checklist de validation

### Import & modèle

- [ ] 4 fichiers importés et renommés (`fact_tickets`, `tickets_risque_scores`, `dim_agents`, `dim_categories`)
- [ ] Auto Date/Time désactivé
- [ ] Table `Calendrier` créée et marquée comme table de dates
- [ ] Colonne calculée `date_created` créée si `created_at` contient une heure
- [ ] Table `_Mesures` créée (colonne Value masquée)
- [ ] 4 relations actives, aucun chemin ambigu
- [ ] Aucune table `LocalDateTable_*` parasite
- [ ] 4 backgrounds PNG importés sur les 4 pages correspondantes

### Mesures DAX créées

- [ ] 15 mesures KPIs_GLOBAUX
- [ ] 6 mesures SLA
- [ ] 7 mesures Agents (dont `Couleur Breach Agent` et `Segment Agent`)
- [ ] 6 mesures Alertes ML

### Valeurs attendues sans filtre (période complète 2022-2024)

| Mesure | Valeur attendue |
|---|---|
| `Nb Tickets Total` | 15 287 |
| `Taux SLA Breach %` | 47,3% |
| `Taux Backlog %` | 7,4% |
| `Nb Backlog` | 1 126 |
| `Delai Premiere Reponse Moy (h)` | 36,8 |
| `CSAT Moyen` | 4,12 |
| `Taux Reouverture %` | 7,7% |
| `Nb Tickets Rouverts` | 1 181 |
| `Taux Breach SLA Strict %` | 42,9% |
| `Depassement SLA Moy (h)` | 121 |
| `Ratio SLA Moyen` | 1,00 |
| `Nb Alertes ROUGE` | 314 |
| `Nb Alertes ORANGE` | 1 542 |
| `Nb Alertes JAUNE` | 931 |
| `Score Risque Max` | 0,50 |
| `Recall Modele` | 79,6% |

### Valeurs catégorie leader

- [ ] Demande info breach = 68,6% (rouge)
- [ ] Facturation breach = 59,4% (rouge)
- [ ] Remboursement breach = 52,6% (rouge)
- [ ] Accès compte breach = 20,7% (vert)

### Valeurs pays leader

- [ ] CI : ~4 000 tickets
- [ ] Sénégal : ~3 000 tickets

### Valeurs agents critiques

- [ ] Moussa Kone breach = 60,1% · 66 alertes ROUGE
- [ ] Ramatou Diaby breach = 55,1% · 52 alertes ROUGE
- [ ] Kofi Mensah breach = 53,6%

### Navigation & slicers

- [ ] 4 boutons transparents posés sur les onglets de chaque PNG
- [ ] Action Navigation de page configurée pour chaque bouton
- [ ] Slicer date début (mode Avant) posé sur la zone du PNG
- [ ] Slicer date fin (mode Après) posé sur la zone du PNG
- [ ] Slicers synchronisés sur les 4 pages

### Pages

- [ ] Page 1 Overview : 6 KPI + courbe SLA + donut statuts + bar catégories + bar pays
- [ ] Page 2 Performance SLA : 4 KPI + jauge + bar catégories + donut + heatmap matrix + bar canaux
- [ ] Page 3 Agents : Tableau 12 agents + scatter Breach×CSAT coloré par Tier
- [ ] Page 4 Alertes ML : 5 KPI + tableau tickets + bar distribution + bar top agents + jauge score

---

## 20. Storytelling — Ordre de présentation (5 minutes)

### Séquence narrative pour M. Kouamé

Structure **Problème → Cause → Solution → Impact**, chaque chiffre suivi de sa signification métier.

#### 1. Overview — *"Voici l'état global du support"* (1 min)

> *"15 287 tickets traités (+24,9% vs l'an dernier). Le taux de SLA breach est de 47,3% — stable vs N-1. L'objectif sectoriel est 30%. On est donc 17 points au-dessus et ce n'est pas une crise ponctuelle. Le CSAT se maintient à 4,12/5, mais le backlog représente 7,4% (1 126 tickets) et 7,7% des tickets sont réouverts (1 181 tickets). Bandeau d'alerte : **3 catégories dépassent 50% de breach** → risque de pénalités contractuelles."*

#### 2. Performance SLA — *"Où explose le SLA ?"* (1 min)

> *"Sur les tickets en breach, le dépassement moyen est de +121h. Les catégories Billing (Demande info 68,6%, Facturation 59,4%, Remboursement 52,6%) concentrent l'essentiel des violations. La heatmap mensuelle confirme : ces catégories sont rouges quasiment toute l'année. En revanche, les 5 canaux sont homogènes à ~47% — le canal n'est PAS un facteur différenciant. C'est donc un problème de **traitement par catégorie**, pas de canal."*

#### 3. Agents — *"Qui performe et qui nécessite un accompagnement ?"* (1 min)

> *"Sur 12 agents, **3 Tier 1 sont en alerte critique** : Moussa Kone 60,1%, Ramatou Diaby 55,1%, Kofi Mensah 53,6% de breach. Leur CSAT est aussi le plus bas (3,5 à 3,8). Les Tier 3 (Jean-Marc Dubois, Olivier Martin) affichent les meilleurs CSAT (4,6-4,8) avec un breach sous 45%. Écart de performance = 20 points de breach et 1,3 point de CSAT → il y a un vrai sujet de **formation/coaching Tier 1**."*

#### 4. Alertes ML — *"La solution préventive est déployée"* (1 min)

> *"Le modèle ML (Recall 79,6%, objectif > 75% ✓) a identifié **314 tickets ROUGE** (intervention immédiate), **1 542 ORANGE** (à surveiller) et **931 JAUNE** (attention). Score de risque moyen 0,50. Les 5 agents Tier 1 déjà identifiés concentrent 214 des 314 alertes ROUGE. En croisant avec leur performance SLA, on a un pattern clair : **ces agents génèrent plus de tickets à risque ET les traitent moins bien**. Le modèle permet au superviseur de basculer le ticket vers un Tier 3 dès sa création."*

#### 5. Plan d'action (1 min)

1. **Coaching 3 agents Tier 1** sous 15 jours → impact estimé -7 points de breach global
2. **Déploiement ML en production** sur dashboard superviseur → évite 80% des breaches futurs par intervention préventive
3. **Renégociation SLA Billing** (3 catégories) sous 60 jours → -8 points de breach mécaniquement

**Objectif 90 jours : ramener le breach de 47% à 25%.**

---

> L'apprenant doit pouvoir répondre à la question : **"Que doit faire AfriCare dans les 90 prochains jours ?"**
>
> La réponse est contenue dans les 4 pages du dashboard — pas ailleurs.

---

## 21. Annexes

### Annexe A — Règles techniques DAX non négociables

- `FILTER(table, [Mesure] = "valeur")` plutôt que `mesure = "valeur"` directement dans `CALCULATE`
- `VAR ... RETURN` pour toutes les mesures avec logique conditionnelle
- `DIVIDE()` pour toutes les divisions (gestion des zéros)
- `IF(ISBLANK(_result), 0, _result)` sur les mesures de comptage qui peuvent être vides
- `PREVIOUSMONTH(Calendrier[Date])` nécessite la table Calendrier marquée comme table de dates
- `RANKX(ALL(dim_agents[agent_id]), ...)` : `ALL` est obligatoire pour ranker sur tous les agents
- `SEARCH("texte", colonne, 1, 0) > 0` pour matcher un texte partiel sans crash sur BLANK
- Pour les flèches dans les pastilles : `UNICHAR(9650)` = ▲, `UNICHAR(9660)` = ▼, `UNICHAR(8594)` = →, `UNICHAR(8212)` = —

### Annexe B — Pièges classiques et solutions

| Piège | Symptôme | Solution |
|---|---|---|
| Auto Date/Time actif | Tables `LocalDateTable_*` parasites | Désactiver dans Options *avant* la modélisation |
| Calendrier non marqué comme date | `SAMEPERIODLASTYEAR` retourne BLANK | Clic droit Calendrier → Marquer comme date table |
| `created_at` avec heure | Relation à Calendrier impossible | Créer colonne calculée `date_created = Date.From([created_at])` |
| RANKX sans `ALL` | Tous les agents au rang 1 | `RANKX(ALL(dim_agents[agent_id]), ...)` obligatoire |
| `SEARCH` sur BLANK | Erreur de calcul | Toujours `SEARCH("texte", col, 1, 0)` (4 args) |
| Mesure de couleur non appliquée | Le visuel reste en couleur par défaut | Format → Couleur → fx → Mettre en forme par : Valeur du champ |
| Mois non triés sur axe X | Janvier après Décembre alphabétique | Outils de colonne → Trier par colonne → Mois_Num |
| Slicer non synchronisé | Filtre actif sur une page mais pas l'autre | Affichage → Synchroniser les segments → cocher Visible ET Filtre |
| Donut tronqué (un secteur disparaît) | Étiquettes de détail trop grandes | Réduire la taille de police ou désactiver les étiquettes intérieures |
| Background flou | Image PNG mal redimensionnée | Format de la page → Ajustement de l'image → Ajuster |



### Annexe C — Mapping rapide visuel ↔ mesures

| Visuel dashboard | Mesures utilisées |
|---|---|
| Card Tickets total | `Nb Tickets Total` |
| Card Taux SLA breach | `Taux SLA Breach %` + `Evolution Breach pp` |
| Card Backlog | `Taux Backlog %` + `Nb Backlog` |
| Card Délai 1ère réponse | `Delai Premiere Reponse Moy (h)` |
| Card CSAT moyen | `CSAT Moyen` |
| Card Réouverture | `Taux Reouverture %` + `Nb Tickets Rouverts` |
| Bar SLA breach par catégorie | `Taux SLA Breach %` colorée par `Couleur Breach Agent` |
| Donut statuts | `fact_tickets[statut]` + `Nb Tickets Total` |
| Bar tickets par pays | `fact_tickets[pays]` + `Nb Tickets Total` (Top 5) |
| Card Breach SLA strict | `Taux Breach SLA Strict %` |
| Card Dépassement | `Depassement SLA Moy (h)` |
| Card Ratio SLA | `Ratio SLA Moyen` |
| Jauge breach | `Taux SLA Breach %` (cible 30%) |
| Heatmap | `Taux SLA Breach %` × catégorie × `Calendrier[Mois_Num]` |
| Bar canal | `Taux SLA Breach %` × `fact_tickets[canal]` |
| Tableau agents — Breach | `Taux Breach Agent %` + `Couleur Breach Agent` |
| Tableau agents — Rang | `Rang Agent SLA` |
| Tableau agents — Écart | `Ecart CSAT Agent` |
| Tableau agents — Segment | `Segment Agent` |
| Scatter Agents | X = `Taux Breach Agent %`, Y = `CSAT Moyen`, Légende = `dim_agents[tier]` |
| Card Alertes ROUGE | `Nb Alertes ROUGE` |
| Card Alertes ORANGE | `Nb Alertes ORANGE` |
| Card Alertes JAUNE | `Nb Alertes JAUNE` |
| Card Score risque | `Score Risque Max` |
| Card Recall | `Recall Modele` |
| Tableau tickets ROUGE | toutes colonnes filtrées niveau_alerte = ROUGE |
| Bar distribution alertes | `tickets_risque_scores[niveau_alerte]` + `Nb Alertes Total` |
| Top 5 agents ROUGE | `dim_agents[nom_agent]` + `Nb Alertes ROUGE` |


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.